In [1]:
import pandas as pd

# ParlaMint-SI analysis dataset preparation
## Merging the initial ParlaSent-SI dataset with aggregated scores

In [ ]:
def process_chunk(df_chunk):
    print(df_chunk.head())
def process_jsonl(file_path, chunksize=1000):
    df_list=[]
    for df_chunk in pd.read_json(file_path, lines=True, chunksize=chunksize):
        df_list.append(df_chunk)
    
    full_df = pd.concat(df_list, ignore_index=True)
    return full_df

file_path = '../../../ParlaSent-SI/ParlaMint-SI.meta.jsonl'
df_sent = process_jsonl(file_path, chunksize=1000)
df_sent.head()

In [ ]:
order = ['ID', 'sent_id', 'text', 'text_en', 'metadata', 'sent_annotations']
df_sent = df_sent.rename(columns={'logit':'sent_annotations', 'newdoc id':'ID'})
df_sent = df_sent[order]
df_sent.head()

In [ ]:
df_utt = pd.read_csv('../../Datasets/ParlaMint-SI/utt_labels_RF.tsv', delimiter='\t', encoding='utf-8')
df_utt.head()

In [5]:
df_utt = df_utt.rename(columns={'annotations':'utt_annotations'})

### Check for no. of utterances

In [ ]:
df1 = set(df_sent['ID'])
df2 = set(df_utt['ID'])

count_df1 = len(df1)
count_df2 = len(df2)

print(f"Number of unique utt_id in df1: {count_df1}")
print(f"Number of unique ID in df2: {count_df2}")

# Identify missing or extra IDs
missing_in_df2 = count_df1 - count_df2
extra_in_df2 = count_df2 - count_df1

print(f"IDs in df1 not in df2: {missing_in_df2}")
print(f"IDs in df2 not in df1: {extra_in_df2}")

### Merging the datasets

In [ ]:
dataset = pd.merge(df_sent, df_utt, on='ID')
dataset

## Adding metadata as columns

In [ ]:
import ast

metadata_df = pd.json_normalize(dataset['metadata'])
dataset = pd.concat([dataset, metadata_df], axis=1).drop(columns=['metadata'])
dataset.head()


In [ ]:
#dataset.to_csv('../../Datasets/Sentiment_Analysis/ParlaMint-SI.tsv', sep='\t', encoding='utf-8', index=False)